# Monitoring check

After the backfill: are the predictions and monitoring tables complete, and are the monitoring numbers right when recomputed by hand?

In [1]:
import json
import pickle
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

pd.set_option("display.width", 200)

monitoring = pd.read_parquet("../datamart/gold/model_monitoring/")
predictions = pd.read_parquet("../datamart/gold/model_predictions/")
features = pd.read_parquet("../datamart/gold/feature_store/")
labels = pd.read_parquet("../datamart/gold/label_store/")
with open("../model_bank/model_selection_2024_03_01.json") as file:
    selection = json.load(file)
print(json.dumps(selection, indent=2))

{
  "train_date": "2024-03-01",
  "selected_model": "credit_model_xgb_2024_03_01",
  "selection_metric": "auc_oot",
  "candidates": {
    "credit_model_logreg_2024_03_01": 0.8409571436940462,
    "credit_model_xgb_2024_03_01": 0.8558266362033797
  }
}


## 1. Predictions: one row per application, from the train date on

In [2]:
scored = predictions.groupby("snapshot_date").size().rename("predictions").to_frame()
scored["applications"] = features.groupby("snapshot_date").size()
print("models:", predictions["model_name"].unique(), "| first month scored:", scored.index.min(),
      "| rows equal applications:", (scored["predictions"] == scored["applications"]).all(),
      "| duplicate keys:", predictions.duplicated(["Customer_ID", "snapshot_date", "model_name"]).sum())
scored.T

models: ['credit_model_xgb_2024_03_01'] | first month scored: 2024-03-01 | rows equal applications: True | duplicate keys: 0


snapshot_date,2024-03-01,2024-04-01,2024-05-01,2024-06-01,2024-07-01,2024-08-01,2024-09-01,2024-10-01,2024-11-01,2024-12-01,2025-01-01
predictions,511,513,491,498,505,543,493,456,488,515,526
applications,511,513,491,498,505,543,493,456,488,515,526


## 2. The monitoring table across time

In [3]:
summary = monitoring[monitoring["metric"].isin(["psi_score", "auc", "gini", "actual_bad_rate", "predicted_bad_rate"])] \
    .pivot_table(index="prediction_month", columns="metric", values="value").round(3)
summary["psi_status"] = monitoring[monitoring["metric"] == "psi_score"].set_index("prediction_month")["status"]
summary["auc_status"] = monitoring[monitoring["metric"] == "auc"].set_index("prediction_month")["status"]
summary["features_red"] = monitoring[monitoring["metric"].str.startswith("csi_") & (monitoring["status"] == "red")].groupby("prediction_month").size()
summary.fillna({"features_red": 0}).astype({"features_red": int})

metric,actual_bad_rate,auc,gini,predicted_bad_rate,psi_score,psi_status,auc_status,features_red
prediction_month,,,,,,,,
2024-03-01,0.329,0.828,0.656,0.305,0.046,green,green,0
2024-04-01,0.277,0.814,0.628,0.283,0.015,green,green,0
2024-05-01,0.303,0.816,0.632,0.288,0.019,green,green,0
2024-06-01,0.311,0.844,0.688,0.273,0.020,green,green,0
2024-07-01,0.307,0.801,0.602,0.240,0.417,red,amber,16
2024-08-01,0.300,0.751,0.502,0.234,0.454,red,red,16
2024-09-01,0.331,0.792,0.585,0.240,0.401,red,amber,16
2024-10-01,0.246,0.788,0.576,0.222,0.585,red,amber,16
2024-11-01,0.281,0.800,0.600,0.239,0.596,red,amber,16


In [4]:
# performance appears only once labels exist: run month = scored month + 6
perf = monitoring[monitoring["metric_group"] == "performance"]
lag = (pd.to_datetime(perf["run_date"]).dt.to_period("M") - pd.to_datetime(perf["prediction_month"]).dt.to_period("M")).apply(lambda d: d.n)
print("months between scoring and measuring performance:", lag.unique(), "| first performance run:", perf["run_date"].min(),
      "| rows in the table:", len(monitoring), "in", monitoring["run_date"].nunique(), "run months")

months between scoring and measuring performance: [6] | first performance run: 2024-09-01 | rows in the table: 715 in 17 run months


**Findings:**
- The model trained at 2024-03-01 (XGBoost, OOT AUC 0.856, chosen over logistic regression's 0.841) scored every application from 2024-03 to 2025-01: one row per application, none earlier, no duplicates.
- 2024-03 to 2024-06: score PSI 0.015–0.046 and AUC 0.814–0.844, all green, and the predicted bad rate within 4 points of the actual.
- 2024-07: the score PSI jumps to 0.417 (red) and all 16 clickstream features turn red, in the month the applications are scored. Their AUC (0.801, amber) is known only six months later, in the 2025-01 run.
- From 2024-07 the AUC is 0.75–0.82: amber in five months, red for 2024-08 (0.751, more than 0.10 below the OOT AUC), green for 2024-12. Stability warned six months before performance confirmed.
- From 2024-07 the model also under-predicts defaults: a predicted bad rate of 0.21–0.24 against an actual 0.25–0.33.
- Performance is always measured 6 months after scoring; the first performance row is in the 2024-09 run.

## 3. Recompute one month by hand: applications of 2024-07

In [5]:
month = pd.Timestamp("2024-07-01").date()
p = predictions[predictions["snapshot_date"] == month]
l = labels[labels["loan_start_date"] == month][["Customer_ID", "label", "snapshot_date"]]
joined = p.merge(l, on="Customer_ID")
auc_by_hand = roc_auc_score(joined["label"], joined["model_predictions"])
auc_in_table = monitoring[(monitoring["prediction_month"] == month) & (monitoring["metric"] == "auc")]["value"].iloc[0]
print(f"label date of these applications: {joined['snapshot_date_y'].unique()} | applications with a label: {len(joined)}\n"
      f"AUC by hand: {auc_by_hand:.6f} | in the monitoring table: {auc_in_table:.6f}")

label date of these applications: [datetime.date(2025, 1, 1)] | applications with a label: 505
AUC by hand: 0.800931 | in the monitoring table: 0.800931


In [6]:
# score PSI by hand, step by step: bins = deciles of the training scores stored in the artefact
with open("../model_bank/" + selection["selected_model"] + ".pkl", "rb") as file:
    artefact = pickle.load(file)
train_scores = pd.Series(artefact["reference"]["train_scores"])
edges = np.unique(np.quantile(train_scores, np.linspace(0.1, 0.9, 9)))
bins = pd.DataFrame({
    "training": np.bincount(np.digitize(train_scores, edges, right=True), minlength=len(edges) + 1) / len(train_scores),
    "2024-07": np.bincount(np.digitize(p["model_predictions"], edges, right=True), minlength=len(edges) + 1) / len(p),
})
bins["contribution"] = (bins["2024-07"] - bins["training"]) * np.log(bins["2024-07"] / bins["training"])
psi_in_table = monitoring[(monitoring["prediction_month"] == month) & (monitoring["metric"] == "psi_score")]["value"].iloc[0]
print("PSI by hand:", round(bins["contribution"].sum(), 6), "| in the monitoring table:", round(psi_in_table, 6))
bins.round(3)

PSI by hand: 0.417405 | in the monitoring table: 0.417405


,training,2024-07,contribution
0,0.1,0.055,0.026
1,0.1,0.145,0.016
2,0.1,0.190,0.058
3,0.1,0.145,0.017
4,0.1,0.077,0.006
5,0.1,0.030,0.085
6,0.1,0.057,0.023
7,0.1,0.139,0.013
8,0.1,0.147,0.018
9,0.1,0.016,0.155


**Findings:**
- AUC (0.800931) and score PSI (0.417405) for the 2024-07 applications, recomputed from the gold predictions, the label store and the model artefact, equal the monitoring table's values to 6 decimals.
- Most of the PSI comes from the top of the score range: 1.6% of July's applicants fall in the highest training decile, against 10% in training. With clickstream missing, the riskiest applicants no longer stand out.

## 4. Which features drift, and when?

In [7]:
csi = monitoring[monitoring["metric"].str.startswith("csi_")]
red = csi[csi["status"] == "red"]
pd.DataFrame({
    "features": csi.groupby("prediction_month").size(),
    "red": red.groupby("prediction_month").size(),
    "of which clickstream": red[red["metric"].str.endswith("_avg_3m")].groupby("prediction_month").size(),
    "largest non-clickstream CSI": csi[~csi["metric"].str.endswith("_avg_3m")].groupby("prediction_month")["value"].max().round(3),
}).fillna(0).astype({"red": int, "of which clickstream": int})

,features,red,of which clickstream,largest non-clickstream CSI
prediction_month,,,,
2024-03-01,59,0,0,0.052
2024-04-01,59,0,0,0.058
2024-05-01,59,0,0,0.073
2024-06-01,59,0,0,0.037
2024-07-01,59,16,16,0.068
2024-08-01,59,16,16,0.062
2024-09-01,59,16,16,0.048
2024-10-01,59,16,16,0.056
2024-11-01,59,16,16,0.053


**Findings:**
- From 2024-07 every red feature is a clickstream feature, all 16 the model uses. The largest CSI among the other 43 features stays between 0.037 and 0.087 in every month, all green. The drift has one cause: the clickstream feed stopped.